# Chapter 6: Agents That Collaborate

This notebook mirrors the chapter. Run the cells in order, from the `Chapter06`
folder. Nothing here needs an API key. Two supplier agents run as real A2A
servers inside this process, with signed Agent Cards and task endpoints, and the
cells call them through the official A2A client. Only the model is replaced, by
the Agents SDK's scripted test model.

Pinned versions: a2a-sdk 1.3.0, openai-agents 0.24.0.

In [1]:
import sys
from importlib.metadata import version
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
for package in ("a2a-sdk", "openai-agents"):
    print(f"{package:14} {version(package)}")

a2a-sdk        1.3.0
openai-agents  0.24.0


## Diplomacy has a protocol: Inside A2A

### Discovering a peer with the Agent Card

Build NordBolt's card, sign it, verify the signature, then change one skill
description and verify again. The changed card must be refused.

In [2]:
from examples import show_card

show_card.main()

{
  "name": "nordbolt-quoting-agent",
  "description": "Quotes industrial fasteners and takes orders against its quotes.",
  "supportedInterfaces": [
    {
      "url": "https://agents.nordbolt.example/a2a",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "provider": {
    "url": "https://agents.nordbolt.example",
    "organization": "NordBolt Industrial Fasteners"
  },
  "version": "1.4.0",
  "capabilities": {
    "streaming": true,
    "pushNotifications": true
  },
  "securitySchemes": {
    "oauth": {
      "oauth2SecurityScheme": {
        "flows": {
          "clientCredentials": {
            "tokenUrl": "https://agents.nordbolt.example/oauth/token",
            "scopes": {
              "quotes:request": "Request quotes and place orders"
            }
          }
        }
      }
    }
  },
  "securityRequirements": [
    {
      "schemes": {
        "oauth": {
          "list": [
            "quotes:request"
          ]
        }
      }
    }


### Opening a task

The task states, as they are written on the wire. Four are terminal. Two park a
task until the caller supplies input or authentication.

In [3]:
from a2a.types import TaskState

for name in TaskState.keys():
    print(name)

TASK_STATE_UNSPECIFIED
TASK_STATE_SUBMITTED
TASK_STATE_WORKING
TASK_STATE_COMPLETED
TASK_STATE_FAILED
TASK_STATE_CANCELED
TASK_STATE_INPUT_REQUIRED
TASK_STATE_REJECTED
TASK_STATE_AUTH_REQUIRED


## Opening the embassy: Exposing your agent over A2A

### Deciding what the card advertises

The agent behind the card is an ordinary Agents SDK agent. Its tools are private.
Its output type is the public contract.

In [4]:
from chapter6.agent import build_quote_agent
from chapter6.contracts import FastenerQuote

agent = build_quote_agent()
print("agent  :", agent.name)
print("tools  :", ", ".join(tool.name for tool in agent.tools))
print("output :", agent.output_type.__name__)
print("fields :", ", ".join(FastenerQuote.model_fields))

agent  : nordbolt_quoting
tools  : check_inventory, price_order
output : FastenerQuote
fields : quote_id, unit_price, currency, lead_time_days, valid_until, conditions


### Bridging the run loop to the task lifecycle, and serving the agent

One round trip: registry lookup, card verification, a quote task through
NordBolt's border control and executor, and a validated artifact. The last three
lines are the boundary audit record.

In [5]:
from examples import quote_roundtrip

await quote_roundtrip.main()

ok          : True
quote       : NB-9A3D2B
unit price  : 3.13 GBP
lead time   : 28 days
valid until : 2026-11-25

audit       : quote with nordbolt, card 1.4.0
              task state completed, authorized by buyer/a.okafor
              request 7418429c0ea7, response ee48dc1cd00e


### Border control

A bill of materials with an instruction hidden in a part description. Read the
`model calls` line: the request is refused before the agent runs.

In [6]:
from drills import hostile_request

await hostile_request.main()

ok          : False
error class : supplier_rejected
retryable   : False
detail      : Rejected at the border: free text looks like an instruction
model calls : 0
quotes sent : 0


## Hiring a stranger: Consuming remote agents

### Finding a remote agent

A registry entry stores the trust relationship, of which the key is one part.

In [7]:
from chapter6.neighborhood import build_neighborhood
from chapter6.registry import CounterpartyRegistry

registry = CounterpartyRegistry()
hood = build_neighborhood(registry)
entry = registry.require("nordbolt")
trust = entry.trust
print("organization :", trust.organization)
print("card         :", trust.card_version, trust.card_fingerprint)
print("trust anchor :", trust.trust_anchor)
print("contract     :", trust.contract_reference)
print("approved by  :", trust.approved_by)
for key in trust.signing_keys:
    print(f"key          : {key.kid}, {key.not_before} to {key.not_after}")
print("wait up to   :", entry.sla_seconds, "seconds")
await hood.aclose()

organization : NordBolt Industrial Fasteners
card         : 1.4.0 ae1123d475e0e1bcfe736c51976aa8dad275624a814b06d100a390de4988cf43
trust anchor : Public key exchanged under MSA-NORDBOLT-2026, schedule C
contract     : MSA-NORDBOLT-2026
approved by  : procurement-security-board
key          : nordbolt-2026-01, 2026-10-12 to 2027-10-12
key          : nordbolt-2025-01, 2025-10-12 to 2026-12-11
wait up to   : 5.0 seconds


The same server presents three cards in turn: the approved card signed with the
previous key, the approved card signed with an unknown key, and a validly signed
card whose content has changed.

In [8]:
from drills import untrusted_card

await untrusted_card.main()

previous key, in date   : quote received
unknown key             : supplier_untrusted_card
                          No valid signature found
valid key, changed card : supplier_untrusted_card
                          card 1.5.0 is not the card that was approved


### Calling a remote agent

A request that leaves out the drawing revision for a custom part. The supplier
parks the task and asks. The answer goes back on the same task.

In [9]:
from drills import parked_task

await parked_task.main()

first answer : supplier_input_required
question     : Which drawing revision applies to CB-42?
model calls  : 0

same task    : True
second answer: quote NB-9A3D2B, 3.13 GBP
model calls  : 2


### Handling remote agent failures

NordBolt goes offline. Three failures open its circuit breaker, the fourth call
is refused locally, and a fan-out still returns the second supplier's quote.

In [10]:
from drills import supplier_down

await supplier_down.main()

call 1: supplier_unreachable   retryable=True  breaker=closed
call 2: supplier_unreachable   retryable=True  breaker=closed
call 3: supplier_unreachable   retryable=True  breaker=open
call 4: supplier_circuit_open  retryable=False breaker=open

fan-out nordbolt   : supplier_circuit_open
fan-out rivetridge : 3.29 GBP


## Putting it together: A supply chain across three organizations

### Running a sourcing round

Fan out, compare in plain code, propose, approve one order at one price, commit,
and print the boundary audit trail. The two quote lines in the trail can appear in
either order, because the requests run in parallel.

In [11]:
from drills import sourcing_run

await sourcing_run.main()

quotes
  nordbolt    3.13 GBP  28 days  NB-9A3D2B
  rivetridge  3.29 GBP  24 days  RR-9A3D2B

proposal   PO-88231 to nordbolt: 2000 x 3.13 = 6,260.00 GBP
approval   APR-5521 by a.okafor, up to 6,260.00, until 2026-11-25
commit     committed=True confirmation=CONF-PO-88231

boundary audit trail
  quote          rivetridge  completed  procurement-spine
  quote          nordbolt    completed  procurement-spine
  commit_intent  nordbolt    -          buyer/a.okafor
  commit_outcome nordbolt    completed  buyer/a.okafor


The commit is retried with the same purchase order number, then attempted with
the same number and a different quantity. One order must exist at the end.

In [12]:
from drills import retry_same_message

await retry_same_message.main()

first commit   : CONF-PO-88231
second commit  : CONF-PO-88231
changed commit : committed=False
  Rejected at the border: message ID reused with different content
orders on the supplier's ledger: 1


### The three-day silence

The webhook handler: early notifications held and released, a duplicate
acknowledged and ignored, and a wrong token refused.

In [13]:
from drills import three_day_silence

await three_day_silence.main()

task parked     : supplier_input_required
held, then freed: 3 notifications


work queue      :
  nordbolt  TASK_STATE_SUBMITTED
  nordbolt  TASK_STATE_WORKING
  nordbolt  TASK_STATE_INPUT_REQUIRED
  nordbolt  TASK_STATE_WORKING
  nordbolt  artifact
  nordbolt  TASK_STATE_COMPLETED

duplicate       : HTTP 204, queue length 6
wrong token     : HTTP 401, queue length 6


## Self-check

The test suite covers the same behavior with assertions. Every test should pass.

In [14]:
import subprocess

done = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no"],
    capture_output=True,
    text=True,
)
print(done.stdout.strip().splitlines()[-1])

38 passed in 1.26s
